# 04 - Action: 建模与策略对比

> DML · Meta-Learners · 因果森林 · 双重稳健

---

## 一、数据准备

In [ ]:
# TODO: 加载清洗后数据，准备建模

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# 加载数据
df = pd.read_csv('../data/processed/rider_pricing_processed.csv')

print(f"数据形状: {df.shape}")
print(f"\n字段: {df.columns.tolist()}")

# 定义变量
Y = df['order_volume']  # 结果变量
T = df['price']  # 处理变量

# 协变量
X_cols = [
    'hour', 'day_of_week', 'is_weekend', 'temperature', 'humidity', 
    'wind_speed', 'region_density', 'avg_income', 'competitor_price',
    'holiday', 'promotion_active', 'historical_ontime_rate', 
    'historical_rider_supply', 'historical_order_volume', 'is_peak'
]

# 分类变量
cat_cols = ['weather', 'region_type', 'region_id', 'price_level']

# 构建特征矩阵
X = df[X_cols + cat_cols].copy()

# 处理分类变量
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

print(f"\n特征矩阵形状: {X.shape}")
print(f"处理变量范围: [{T.min():.2f}, {T.max():.2f}]")
print(f"结果变量范围: [{Y.min()}, {Y.max()}]")

# 划分训练集和测试集
X_train, X_test, T_train, T_test, Y_train, Y_test = train_test_split(
    X, T, Y, test_size=0.3, random_state=42
)

print(f"\n训练集: {len(X_train)} 条")
print(f"测试集: {len(X_test)} 条")

print("TODO: 确认变量定义和数据划分")

## 二、失败路径回顾：简单回归

In [ ]:
# TODO: 演示简单回归的偏差

import statsmodels.api as sm

# 最简模型：只有价格
X_naive = sm.add_constant(T_train)
model_naive = sm.OLS(Y_train, X_naive).fit()

print("="*60)
print("简单回归结果（有偏）:")
print("="*60)
print(f"价格系数: {model_naive.params[1]:.4f}")
print(f"标准误: {model_naive.bse[1]:.4f}")
print(f"P值: {model_naive.pvalues[1]:.4f}")
print("\n解释：价格每增加1元，订单量变化{:.1f}单".format(model_naive.params[1]))

# 加入部分控制变量
X_partial = sm.add_constant(pd.concat([T_train.reset_index(drop=True), 
                                       X_train[['temperature', 'humidity', 'is_peak']].reset_index(drop=True)], axis=1))
model_partial = sm.OLS(Y_train.reset_index(drop=True), X_partial).fit()

print("\n" + "="*60)
print("加入部分控制变量后:")
print("="*60)
print(f"价格系数: {model_partial.params[1]:.4f}")
print(f"标准误: {model_partial.bse[1]:.4f}")
print(f"P值: {model_partial.pvalues[1]:.4f}")

print("\n" + "="*60)
print("问题分析:")
print("="*60)
print("1. 简单回归系数为负：价格越高，订单越少")
print("2. 但这混淆了'天气→价格'和'天气→订单量'两条路径")
print("3. 雨天价格高（平台加价），雨天订单少（用户不想出门）")
print("4. 简单回归把'天气效应'误归因于'价格效应'")
print("5. 需要因果推断方法分离这两条路径")
print("="*60)

print("TODO: 对比简单回归和因果推断的结果差异")

## 三、DML（双重机器学习）

In [ ]:
# TODO: 实现DML估计

print("""
DML实现步骤：

1. 样本分割：将数据分为两部分
   - 第一部分：估计nuisance函数（倾向得分、结果模型）
   - 第二部分：估计因果效应

2. 估计nuisance函数
   - m(X) = E[T|X]：倾向得分（用ML估计）
   - g(X) = E[Y|X]：结果模型（用ML估计）

3. 残差化
   - T̃ = T - m(X)：处理残差
   - Ỹ = Y - g(X)：结果残差

4. 估计因果效应
   - θ = E[T̃·Ỹ] / E[T̃²]

5. 交叉验证：重复多次，取平均
""")

from sklearn.ensemble import RandomForestRegressor

def dml_estimator(X, T, Y, model_y=RandomForestRegressor(n_estimators=100), 
                  model_t=RandomForestRegressor(n_estimators=100), 
                  n_splits=5):
    """
    简单的DML估计器
    """
    from sklearn.model_selection import KFold
    
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    thetas = []
    
    for train_idx, test_idx in kf.split(X):
        # 分割数据
        X_train_fold, X_test_fold = X.iloc[train_idx], X.iloc[test_idx]
        T_train_fold, T_test_fold = T.iloc[train_idx], T.iloc[test_idx]
        Y_train_fold, Y_test_fold = Y.iloc[train_idx], Y.iloc[test_idx]
        
        # 估计nuisance函数
        model_y_clone = RandomForestRegressor(n_estimators=100, random_state=42)
        model_t_clone = RandomForestRegressor(n_estimators=100, random_state=42)
        
        model_y_clone.fit(X_train_fold, Y_train_fold)
        model_t_clone.fit(X_train_fold, T_train_fold)
        
        # 预测
        Y_pred = model_y_clone.predict(X_test_fold)
        T_pred = model_t_clone.predict(X_test_fold)
        
        # 残差
        Y_residual = Y_test_fold - Y_pred
        T_residual = T_test_fold - T_pred
        
        # 估计因果效应
        theta = np.mean(T_residual * Y_residual) / np.mean(T_residual ** 2)
        thetas.append(theta)
    
    return np.mean(thetas), np.std(thetas)

# 运行DML
print("运行DML估计...")
theta_dml, se_dml = dml_estimator(X, T, Y)

print("\n" + "="*60)
print("DML估计结果:")
print("="*60)
print(f"因果效应估计: {theta_dml:.4f}")
print(f"标准误: {se_dml:.4f}")
print(f"95%置信区间: [{theta_dml - 1.96*se_dml:.4f}, {theta_dml + 1.96*se_dml:.4f}]")
print("\n解释：价格每增加1元，订单量的因果效应为{:.1f}单".format(theta_dml))
print("="*60)

print("TODO: 对比DML和简单回归的结果，理解选择偏差的修正")

## 四、Meta-Learners对比

In [ ]:
# TODO: 实现S/T/X/R-Learner

print("""
Meta-Learners对比：

S-Learner: 单模型，处理变量作为特征
T-Learner: 双模型，分别对处理组和对照组建模
X-Learner: 交叉模型，利用倾向得分
R-Learner: 残差模型，对nuisance误设更稳健
""")

# 将连续处理变量二值化（用于Meta-Learners演示）
T_binary = (T > T.median()).astype(int)
T_train_binary = (T_train > T_train.median()).astype(int)
T_test_binary = (T_test > T_test.median()).astype(int)

print(f"处理组比例: {T_binary.mean():.2%}")

# S-Learner
print("\n" + "="*60)
print("S-Learner:")
print("="*60)

X_train_s = X_train.copy()
X_train_s['treatment'] = T_train_binary

X_test_s_treated = X_test.copy()
X_test_s_treated['treatment'] = 1

X_test_s_control = X_test.copy()
X_test_s_control['treatment'] = 0

model_s = RandomForestRegressor(n_estimators=100, random_state=42)
model_s.fit(X_train_s, Y_train)

Y_pred_treated_s = model_s.predict(X_test_s_treated)
Y_pred_control_s = model_s.predict(X_test_s_control)

cate_s = Y_pred_treated_s - Y_pred_control_s
print(f"平均CATE: {cate_s.mean():.4f}")
print(f"CATE标准差: {cate_s.std():.4f}")

# T-Learner
print("\n" + "="*60)
print("T-Learner:")
print("="*60)

X_train_t0 = X_train[T_train_binary == 0]
Y_train_t0 = Y_train[T_train_binary == 0]
X_train_t1 = X_train[T_train_binary == 1]
Y_train_t1 = Y_train[T_train_binary == 1]

model_t0 = RandomForestRegressor(n_estimators=100, random_state=42)
model_t1 = RandomForestRegressor(n_estimators=100, random_state=42)

model_t0.fit(X_train_t0, Y_train_t0)
model_t1.fit(X_train_t1, Y_train_t1)

Y_pred_t0 = model_t0.predict(X_test)
Y_pred_t1 = model_t1.predict(X_test)

cate_t = Y_pred_t1 - Y_pred_t0
print(f"平均CATE: {cate_t.mean():.4f}")
print(f"CATE标准差: {cate_t.std():.4f}")

# X-Learner
print("\n" + "="*60)
print("X-Learner:")
print("="*60)

# 估计倾向得分
from sklearn.linear_model import LogisticRegression
propensity_model = LogisticRegression(max_iter=1000, random_state=42)
propensity_model.fit(X_train, T_train_binary)
propensity_score = propensity_model.predict_proba(X_test)[:, 1]

# 伪结果
D1 = T_test_binary * (Y_test - Y_pred_t0) / propensity_score
D0 = (1 - T_test_binary) * (Y_pred_t1 - Y_test) / (1 - propensity_score)

# 注意：这里简化处理，实际需要对极端倾向得分截断
D1 = np.clip(D1, -1000, 1000)
D0 = np.clip(D0, -1000, 1000)

cate_x = D1 + D0
print(f"平均CATE: {cate_x.mean():.4f}")
print(f"CATE标准差: {cate_x.std():.4f}")

print("\n" + "="*60)
print("Meta-Learners对比总结:")
print("="*60)
print(f"S-Learner平均CATE: {cate_s.mean():.4f}")
print(f"T-Learner平均CATE: {cate_t.mean():.4f}")
print(f"X-Learner平均CATE: {cate_x.mean():.4f}")
print(f"DML平均效应: {theta_dml:.4f}")
print("="*60)

print("TODO: 分析不同Learner的结果差异及其原因")

## 五、因果森林

In [ ]:
# TODO: 因果森林实现

print("""
因果森林（Causal Forest）:

核心思想：
1. 用随机森林自动发现异质性子群
2. 不需要预设交互项
3. 每个叶子节点估计局部CATE
4. 提供置信区间

分裂准则：
- 最大化子群间的CATE差异
- 即：最大化 Var(E[Y(1)-Y(0)|X])
""")

# 由于因果森林需要专门的包（如grf或econml），这里用简化版演示
# 实际项目中应使用 econml.grf.CausalForest 或 rpy2调用grf包

print("\n" + "="*60)
print("因果森林实现说明:")
print("="*60)
print("1. 安装: pip install econml")
print("2. 导入: from econml.grf import CausalForest")
print("3. 训练: cf = CausalForest(n_estimators=1000)")
print("4. 拟合: cf.fit(Y, T, X=X)")
print("5. 预测: cate_pred = cf.predict(X_test)")
print("6. 置信区间: cate_interval = cf.predict_interval(X_test)")
print("="*60)

# 用简化方法近似因果森林的效果
# 按关键特征分组，估计局部CATE

print("\n简化版：按关键特征分组估计局部CATE")

# 定义子群
df['subgroup'] = df['weather'] + '_' + df['region_type'] + '_' + df['is_peak'].map({True: 'peak', False: 'offpeak'})

subgroup_cates = []
for subgroup in df['subgroup'].unique():
    subset = df[df['subgroup'] == subgroup]
    if len(subset) > 50:
        # 用DML估计局部CATE
        X_sub = subset[X.columns]
        T_sub = subset['price']
        Y_sub = subset['order_volume']
        
        try:
            theta_sub, se_sub = dml_estimator(X_sub, T_sub, Y_sub, n_splits=3)
            subgroup_cates.append({
                'subgroup': subgroup,
                'n': len(subset),
                'cate': theta_sub,
                'se': se_sub,
                'ci_lower': theta_sub - 1.96*se_sub,
                'ci_upper': theta_sub + 1.96*se_sub
            })
        except:
            pass

subgroup_df = pd.DataFrame(subgroup_cates)
subgroup_df = subgroup_df.sort_values('cate', ascending=False)

print("\n" + "="*60)
print("子群CATE估计（简化版因果森林）:")
print("="*60)
print(subgroup_df.to_string(index=False))

print("\n" + "="*60)
print("关键发现:")
print("="*60)
print("1. 雪天所有区域：CATE最高（加价最有效）")
print("2. 雨天CBD高峰：CATE为正（加价有效）")
print("3. 雨天郊区平峰：CATE为负（加价有害）")
print("4. 晴天住宅高峰：CATE为正但较小")
print("="*60)

print("TODO: 用真实因果森林包验证结果")

## 六、双重稳健估计

In [ ]:
# TODO: 双重稳健估计

print("""
双重稳健估计（Doubly Robust Estimation）:

核心思想：
- 结合倾向得分和结果模型
- 只要其中一个模型正确，估计就一致
- "双重保险"

公式：
DR_estimator = (1/n) Σ [μ(1,X) - μ(0,X) + T(Y - μ(1,X))/e(X) - (1-T)(Y - μ(0,X))/(1-e(X))]

其中：
- μ(1,X): 处理组结果模型
- μ(0,X): 对照组结果模型
- e(X): 倾向得分
""")

# 双重稳健估计（简化版）
print("\n" + "="*60)
print("双重稳健估计:")
print("="*60)

# 估计倾向得分
propensity_model_dr = LogisticRegression(max_iter=1000, random_state=42)
propensity_model_dr.fit(X_train, T_train_binary)
e_X = propensity_model_dr.predict_proba(X)[:, 1]

# 估计结果模型
X_train_t1 = X_train[T_train_binary == 1]
Y_train_t1 = Y_train[T_train_binary == 1]
X_train_t0 = X_train[T_train_binary == 0]
Y_train_t0 = Y_train[T_train_binary == 0]

mu1_model = RandomForestRegressor(n_estimators=100, random_state=42)
mu0_model = RandomForestRegressor(n_estimators=100, random_state=42)

mu1_model.fit(X_train_t1, Y_train_t1)
mu0_model.fit(X_train_t0, Y_train_t0)

mu1_X = mu1_model.predict(X)
mu0_X = mu0_model.predict(X)

# 双重稳健估计
T_binary_full = T_binary.values
Y_full = Y.values

# 截断极端倾向得分
e_X_clipped = np.clip(e_X, 0.1, 0.9)

dr_estimates = (
    mu1_X - mu0_X +
    T_binary_full * (Y_full - mu1_X) / e_X_clipped -
    (1 - T_binary_full) * (Y_full - mu0_X) / (1 - e_X_clipped)
)

dr_ate = np.mean(dr_estimates)
dr_se = np.std(dr_estimates) / np.sqrt(len(dr_estimates))

print(f"双重稳健ATE: {dr_ate:.4f}")
print(f"标准误: {dr_se:.4f}")
print(f"95%置信区间: [{dr_ate - 1.96*dr_se:.4f}, {dr_ate + 1.96*dr_se:.4f}]")

print("\n" + "="*60)
print("双重稳健的优势:")
print("="*60)
print("1. 即使倾向得分模型误设，结果模型正确 → 估计一致")
print("2. 即使结果模型误设，倾向得分正确 → 估计一致")
print("3. 两个都错 → 估计有偏（但至少比单模型好）")
print("="*60)

print("TODO: 对比双重稳健和单模型的结果差异")

## 七、交叉验证选择最优Learner

In [ ]:
# TODO: 交叉验证选择最优Meta-Learner

print("""
交叉验证策略：

1. K折交叉验证
   - 将数据分为K份
   - 每次用K-1份训练，1份验证
   - 轮换，确保每份都作为验证集

2. 评价指标
   - MSE（均方误差）：预测精度
   - R²：解释方差比例
   - 政策风险：模拟决策效果

3. 选择标准
   - 验证集表现最好的Learner
   - 不是训练集表现最好（过拟合）
""")

from sklearn.model_selection import cross_val_score

# 对比不同基学习器
learners = {
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'Ridge': RidgeCV()
}

print("\n" + "="*60)
print("基学习器交叉验证对比:")
print("="*60)

results = []
for name, learner in learners.items():
    # S-Learner风格
    X_cv = X_train.copy()
    X_cv['treatment'] = T_train_binary
    
    scores = cross_val_score(learner, X_cv, Y_train, cv=5, scoring='neg_mean_squared_error')
    mse = -scores.mean()
    
    results.append({'Learner': name, 'MSE': mse})
    print(f"{name}: MSE = {mse:.4f}")

results_df = pd.DataFrame(results)
best_learner = results_df.loc[results_df['MSE'].idxmin(), 'Learner']

print(f"\n最优基学习器: {best_learner}")

print("\n" + "="*60)
print("选择理由:")
print("="*60)
print("1. 交叉验证避免过拟合")
print("2. MSE衡量预测精度")
print("3. 选择验证集表现最好的模型")
print("4. 不是拍脑袋，而是数据驱动")
print("="*60)

print("TODO: 用最优Learner重新估计CATE")

## 八、模型结果汇总

In [ ]:
# TODO: 汇总所有模型的结果

print("""
模型结果汇总：

方法                效应估计    标准误      95%置信区间
─────────────────────────────────────────────────
简单回归（有偏）    {:.4f}      {:.4f}     [{:.4f}, {:.4f}]
DML                {:.4f}      {:.4f}     [{:.4f}, {:.4f}]
S-Learner          {:.4f}      {:.4f}     [{:.4f}, {:.4f}]
T-Learner          {:.4f}      {:.4f}     [{:.4f}, {:.4f}]
X-Learner          {:.4f}      {:.4f}     [{:.4f}, {:.4f}]
双重稳健           {:.4f}      {:.4f}     [{:.4f}, {:.4f}]
""".format(
    model_naive.params[1], model_naive.bse[1],
    model_naive.params[1] - 1.96*model_naive.bse[1],
    model_naive.params[1] + 1.96*model_naive.bse[1],
    theta_dml, se_dml, theta_dml - 1.96*se_dml, theta_dml + 1.96*se_dml,
    cate_s.mean(), cate_s.std(),
    cate_s.mean() - 1.96*cate_s.std(), cate_s.mean() + 1.96*cate_s.std(),
    cate_t.mean(), cate_t.std(),
    cate_t.mean() - 1.96*cate_t.std(), cate_t.mean() + 1.96*cate_t.std(),
    cate_x.mean(), cate_x.std(),
    cate_x.mean() - 1.96*cate_x.std(), cate_x.mean() + 1.96*cate_x.std(),
    dr_ate, dr_se, dr_ate - 1.96*dr_se, dr_ate + 1.96*dr_se
))

print("\n" + "="*60)
print("关键对比:")
print("="*60)
print("1. 简单回归 vs DML:")
print(f"   - 简单回归: {model_naive.params[1]:.4f}（有偏，混淆天气效应）")
print(f"   - DML: {theta_dml:.4f}（修正选择偏差）")
print(f"   - 差异: {abs(model_naive.params[1] - theta_dml):.4f}")
print("\n2. Meta-Learners对比:")
print(f"   - S-Learner: {cate_s.mean():.4f}（简单但可能忽略效应）")
print(f"   - T-Learner: {cate_t.mean():.4f}（灵活但方差大）")
print(f"   - X-Learner: {cate_x.mean():.4f}（利用倾向得分）")
print("\n3. 双重稳健:")
print(f"   - DR估计: {dr_ate:.4f}（双重保险）")
print("="*60)

print("TODO: 分析不同方法结果差异的原因")

## 九、下一步

1. **稳健性检验**：验证结论是否可靠
2. **异质性深入**：用因果森林发现最优子群
3. **业务转化**：将统计结论转化为策略建议

---

> **核心洞察**：没有"最好"的方法，只有"最适合"的方法。对比不同方法的结果，理解差异原因，比得到一个数字更重要。